# ML-04 — Search Intelligence Data Contract

*   List item
*   List item



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityaram738/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Unit of analysis

- One row represents one unique client.
- The table contains one record per client.
- The time-related fields describe the client's lifecycle and data availability:
  - `client_created_date`
  - `client_updated_date`
  - `gsc_data_start`
  - `ga4_data_start`
- This is a client dimension table rather than a time-series table, so Each row represents a single client. The table is not time-series data; the date fields record client lifecycle milestones (client_created_date, client_updated_date, gsc_data_start, ga4_data_start) rather than a common reporting window.

In [3]:
con.execute("""
DESCRIBE
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
""").fetchdf()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


## 2. Fields: feature / label / context / excluded

### Features
- is_active
- has_gsc_access
- has_ga4_access

### Label
- None (this table does not contain a prediction target)

### Context
- access_profile
- client_created_date
- client_updated_date
- gsc_data_start
- ga4_data_start

### Excluded
- client_hash_id (identifier only; it uniquely identifies a client and should not be used as a predictive feature)

In [4]:
con.execute("""
SELECT
    COUNT(*) AS total_clients,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    SUM(CASE WHEN is_active THEN 1 ELSE 0 END) AS active_clients,
    SUM(CASE WHEN has_gsc_access THEN 1 ELSE 0 END) AS gsc_access_clients,
    SUM(CASE WHEN has_ga4_access THEN 1 ELSE 0 END) AS ga4_access_clients
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
""").fetchdf()

,total_clients,unique_clients,active_clients,gsc_access_clients,ga4_access_clients
0,104,104,74.0,67.0,54.0


## 3. Verify it with queries (grain, counts, missing values, windows)

### Verification Summary

- The dataset contains 104 rows and 104 unique clients.
- Each row represents one client.
- The identifier (`client_hash_id`) is unique.
- This table has no prediction label.
- Boolean fields indicate client status and product access.
- Date fields provide context about client lifecycle and data availability.
- Missing values should be checked before using the data for analysis or modeling.

In [5]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS unique_clients,

    SUM(CASE WHEN client_hash_id IS NULL THEN 1 ELSE 0 END) AS missing_client_hash_id,
    SUM(CASE WHEN is_active IS NULL THEN 1 ELSE 0 END) AS missing_is_active,
    SUM(CASE WHEN has_gsc_access IS NULL THEN 1 ELSE 0 END) AS missing_has_gsc_access,
    SUM(CASE WHEN has_ga4_access IS NULL THEN 1 ELSE 0 END) AS missing_has_ga4_access,
    SUM(CASE WHEN access_profile IS NULL THEN 1 ELSE 0 END) AS missing_access_profile,
    SUM(CASE WHEN client_created_date IS NULL THEN 1 ELSE 0 END) AS missing_client_created_date,
    SUM(CASE WHEN client_updated_date IS NULL THEN 1 ELSE 0 END) AS missing_client_updated_date,
    SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS missing_gsc_data_start,
    SUM(CASE WHEN ga4_data_start IS NULL THEN 1 ELSE 0 END) AS missing_ga4_data_start
FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
""").fetchdf()

,total_rows,unique_clients,missing_client_hash_id,missing_is_active,missing_has_gsc_access,missing_has_ga4_access,missing_access_profile,missing_client_created_date,missing_client_updated_date,missing_gsc_data_start,missing_ga4_data_start
0,104,104,0.0,10.0,10.0,10.0,0.0,10.0,10.0,37.0,53.0


## 4. Data limits

### Data Limits

- This dataset contains client metadata, not website performance metrics.
- It cannot measure traffic, clicks, impressions, conversions, or revenue.
- Some clients do not have GSC or GA4 connected, so data availability is incomplete.
- Several fields contain missing values and require null handling before analysis.
- The table has no prediction label, so it cannot be used alone to train a supervised machine learning model.
- This table should be joined with other warehouse tables for richer analysis.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.